# Generate `axion_detector_B_scan.png` with panel labels (a)-(d)

Runs the QuTiP magnetic-field scan from
`axion_rydberg_detector_magnetic_field` and overlays panel labels
**(a)**, **(b)**, **(c)**, **(d)** next to each subplot title.
The simulation itself is unchanged with respect to the version
that produced the figure currently committed in
`PhDThesis/chapters/qs_figures/`, except for one physically
motivated correction (laser auto-detuning, see section 2).

## Prerequisites

- **QuTiP** $\geq$ 5.0 (tested on 5.2.3)
- `numpy`, `matplotlib`, internet access (for the GitHub fallback)

## Portability

The notebook is **directory-agnostic**: it looks for the package
source in a few common locations and, if it cannot find it,
downloads it from GitHub (`rcapp2506/qh_sensing`, branch `main`)
into a temporary directory.

## Runtime

With the laser auto-detuning correction applied (see section 2),
approximately 30&ndash;60 seconds total.

## 1. Locate (or download) the package source

In [ ]:
import os
import sys
import tempfile

MODULE_NAME = 'axion_rydberg_detector_magnetic_field'
MODULE_FILE = MODULE_NAME + '.py'

GITHUB_RAW_URL = (
    'https://raw.githubusercontent.com/rcapp2506/qh_sensing/main/'
    '3_qutip_simpulation_with_magnetic_field/'
    'axion_rydberg_package/code/' + MODULE_FILE
)

_here = os.path.abspath(os.getcwd())
print(f'Notebook cwd: {_here}')

candidate_dirs = [
    os.path.normpath(os.path.join(_here, '..', 'code')),
    os.path.normpath(os.path.join(_here, 'code')),
    _here,
]

found_dir = None
for d in candidate_dirs:
    if os.path.isfile(os.path.join(d, MODULE_FILE)):
        found_dir = d
        break

if found_dir is not None:
    print(f'Found {MODULE_FILE} in: {found_dir}')
    if found_dir not in sys.path:
        sys.path.insert(0, found_dir)
else:
    print(f'Will download {MODULE_FILE} from GitHub in the next cell.')

### 1b. Fallback: download from GitHub if not found locally

In [ ]:
if found_dir is None:
    tmp_dir = tempfile.mkdtemp(prefix='axion_rydberg_')
    tgt = os.path.join(tmp_dir, MODULE_FILE)
    print(f'Downloading {GITHUB_RAW_URL}')
    print(f'        ->  {tgt}')

    try:
        import requests
        r = requests.get(GITHUB_RAW_URL, timeout=30)
        r.raise_for_status()
        with open(tgt, 'wb') as f:
            f.write(r.content)
    except ImportError:
        import urllib.request
        urllib.request.urlretrieve(GITHUB_RAW_URL, tgt)

    size_kb = os.path.getsize(tgt) / 1024
    print(f'Downloaded {size_kb:.1f} kB')

    sys.path.insert(0, tmp_dir)
    found_dir = tmp_dir

print(f'Package source directory: {found_dir}')

## 2. Import dependencies

After commit XXX upstream (qh_sensing) the `Delta_gr = -V_rr` auto-detuning
is implemented directly in `RydbergAxionDetectorParams.__init__`, so the
monkey-patches that previous notebook revisions installed at this stage
are no longer needed and have been removed.


In [ ]:
import qutip
print(f'QuTiP version: {qutip.__version__}')


## 3. Import the package


In [ ]:
from axion_rydberg_detector_magnetic_field import (
    scan_magnetic_field,
    plot_magnetic_field_comparison,
    RydbergAxionDetectorParams,
)

# Sanity check: confirm that the in-package auto-detuning yields
# exact facilitation Delta_gr + V_rr = 0 at every B by construction.
import numpy as np
print('In-package auto-detuning sanity check (no monkey-patch needed):')
print()
print(f'  B (T) | Delta_gr/(2pi) (MHz) | V_rr/(2pi) (MHz) | residual (MHz)')
for B in [0.0, 1.0, 3.0, 5.0]:
    p = RydbergAxionDetectorParams(B_field=B, N=11)
    res = (p.Delta_gr + p.V_rr) / (2*np.pi) / 1e6
    print(f'  {B:.1f}   |  {p.Delta_gr/(2*np.pi)/1e6:>16.3f}    |  {p.V_rr/(2*np.pi)/1e6:>12.3f}    |  {res:.2e}')


## 4. Run the magnetic-field scan

Four QuTiP integrations at $B = 0, 1, 3, 5$ T, $N = 11$ atoms,
$T = 4$ K. The progress bar reports per-B-value progress.
Expected total runtime: $\sim 30$&ndash;$60$ seconds.

In [ ]:
results = scan_magnetic_field(
    B_values=[0.0, 1.0, 3.0, 5.0],
    temperature=4.0,
    N=11,
)

## 5. Generate the standard 4-panel figure

In [ ]:
fig = plot_magnetic_field_comparison(results, save_fig=False)
print(f'Figure has {len(fig.axes)} axes total '
      f'(first 4 are data panels, the rest are colorbars)')

## 6. Re-style panel (a) with a high-contrast palette

The default `viridis` ramp sampled at 4 points produces two nearly
identical greens for $B = 1$ T and $B = 3$ T, so the four traces
are visually inseparable. We replace the colors with a qualitative
palette (navy / magenta / forest green / dark orange) and assign a
distinct linestyle to each trace so the curves remain
distinguishable even where they overlap.

In [ ]:
ax_a = fig.axes[0]
B_sorted = sorted(results.keys())

restyle = [
    ('#003f7f', '-',  2.5),   # B = 0 T : navy blue,    solid
    ('#c71585', '--', 2.5),   # B = 1 T : magenta,      dashed
    ('#228b22', '-.', 2.5),   # B = 3 T : forest green, dash-dot
    ('#ff8c00', ':',  3.0),   # B = 5 T : dark orange,  dotted
]

for line, B, (color, ls, lw) in zip(ax_a.get_lines(), B_sorted, restyle):
    line.set_color(color)
    line.set_linestyle(ls)
    line.set_linewidth(lw)
    line.set_alpha(1.0)

ax_a.legend(fontsize=10, loc='lower right')
fig

## 7. Overlay panel labels **(a)**, **(b)**, **(c)**, **(d)**

Non-bold labels, same fontsize as the panel title, placed at the
left edge of each subplot. A cleanup step at the top removes any
leftover labels from previous runs (so the cell can be re-executed
without duplicating the labels).

In [ ]:
# Cleanup: remove any leftover label artists from previous runs.
LABEL_TEXTS = {'(a)', '(b)', '(c)', '(d)'}
for ax in fig.axes[:4]:
    for txt in list(ax.texts):
        if txt.get_text().strip() in LABEL_TEXTS:
            txt.remove()

# Place the labels next to each panel title (left-aligned, non-bold).
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for ax, lbl in zip(fig.axes[:4], panel_labels):
    title = ax.get_title()
    title_fontsize = ax.title.get_fontsize()
    ax.set_title('')
    ax.text(
        0.0, 1.02,
        f'{lbl}  {title}',
        transform=ax.transAxes,
        fontsize=title_fontsize,
        fontweight='normal',
        va='bottom', ha='left',
    )

fig

## 8. Save with the manuscript-expected filename

In [ ]:
out_path = os.path.join(_here, 'axion_detector_B_scan.png')
fig.savefig(out_path, dpi=300, bbox_inches='tight')

print(f'Saved: {out_path}')
print(f'File size: {os.path.getsize(out_path) / 1024:.1f} kB')

## 9. (Optional) Verify the saved PNG

In [ ]:
from IPython.display import Image, display
display(Image(filename=out_path))

## Next steps

1. Copy the generated PNG into the manuscript:
   ```bash
   cp axion_detector_B_scan.png \
      <path>/PhDThesis/chapters/qs_figures/
   ```

2. Verify the LaTeX compiles cleanly:
   ```bash
   cd <path>/PhDThesis
   pdflatex -interaction=nonstopmode main.tex
   ```

3. Commit and push:
   ```bash
   git add chapters/qs_figures/axion_detector_B_scan.png
   git commit -m "Cap. 4 Fig 4.1: add panel labels (a)-(d), regenerate PNG"
   git push origin main
   ```